In [19]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

In [20]:
url = "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result" 
# just testing on one race first - 2023 worlds 100m final


resp = requests.get(url, headers=HEADERS, timeout=15) 
resp.raise_for_status() # will error out if the page didn't load properly (404 etc)

tables = pd.read_html(StringIO(resp.text))
print(f"Found {len(tables)} tables on the page")

results_table = max(tables, key=len) # results table is usually gonna be the biggest one on the page
results_table

Found 1 tables on the page


,POS,BIB,ATHLETE,COUNTRY,MARK,ms*,Reaction Time
0,1,3007,Noah LYLES,USA,9.83 =WL,NaN,0.145
1,2,2102,Letsile TEBOGO,BOT,9.88 NR,0.873,0.129
2,3,2350,Zharnel HUGHES,GBR,9.88,0.874,0.105
3,4,2554,Oblique SEVILLE,JAM,9.88,0.877,0.148
4,5,2978,Christian COLEMAN,USA,9.92,NaN,0.137
5,6,2587,Abdul Hakim SANI BROWN,JPN,10.04,NaN,0.177
6,7,2626,Ferdinand OMANYALA,KEN,10.07,NaN,0.152
7,8,2540,Ryiem FORDE,JAM,10.08,NaN,0.188


In [21]:
import re  # for pulling the wind number out of a sentence

soup = BeautifulSoup(resp.text, "html.parser")

heading = soup.find(string=lambda s: s and "Wind" in s) # search all the text on the page for anything containing "Wind"
print(heading) 



                        Wind 0.0
                


In [22]:
# now grab just the number after "Wind"
wind = None
if heading:
    match = re.search(r"Wind\s*(-?\d+\.\d+)", heading)
    if match:
        wind = float(match.group(1))

print("wind:", wind)

wind: 0.0


In [23]:
import numpy as np

def clean_mark(mark):
    """Split '9.79 PB' into (9.79, 'PB'), or '9.91' into (9.91, None)"""
    if pd.isna(mark):
        return np.nan, None
    parts = str(mark).strip().split()
    time = float(parts[0])
    flag = parts[1] if len(parts) > 1 else None
    return time, flag

results_table[["time", "record_flag"]] = results_table["MARK"].apply(
    lambda m: pd.Series(clean_mark(m))
)

results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time"]]

,ATHLETE,COUNTRY,time,record_flag,Reaction Time
0,Noah LYLES,USA,9.83,=WL,0.145
1,Letsile TEBOGO,BOT,9.88,NR,0.129
2,Zharnel HUGHES,GBR,9.88,NaN,0.105
3,Oblique SEVILLE,JAM,9.88,NaN,0.148
4,Christian COLEMAN,USA,9.92,NaN,0.137
5,Abdul Hakim SANI BROWN,JPN,10.04,NaN,0.177
6,Ferdinand OMANYALA,KEN,10.07,NaN,0.152
7,Ryiem FORDE,JAM,10.08,NaN,0.188


In [ ]:
def scrape_race(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    
    tables = pd.read_html(StringIO(resp.text))
    results_table = max(tables, key=len)
    
    results_table[["time", "record_flag"]] = results_table["MARK"].apply(
        lambda m: pd.Series(clean_mark(m))
    )
    
    soup = BeautifulSoup(resp.text, "html.parser")
    heading = soup.find(string=lambda s: s and "Wind" in s)
    print(f"DEBUG heading for {url[:50]}...: {heading!r}")   # <-- add this
    
    wind = None
    if heading:
        match = re.search(r"Wind\s*(-?\d+\.\d+)", heading)
        if match:
            wind = float(match.group(1))
    
    results_table["wind"] = wind
    results_table["source_url"] = url
    
    return results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time", "wind", "source_url"]]

In [25]:
race1 = scrape_race("https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result")
race2 = scrape_race(url)  # whichever URL gave you the Lyles 9.83 result

pd.concat([race1, race2])

C:\Users\rotem\AppData\Local\Temp\ipykernel_2672\580866877.py:4: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([race1, race2])


,ATHLETE,COUNTRY,time,record_flag,Reaction Time,wind,source_url
0,Noah LYLES,USA,9.79,PB,0.178,NaN,https://worldathletics.org/results/olympic-gam...
1,Kishane THOMPSON,JAM,9.79,NaN,0.176,NaN,https://worldathletics.org/results/olympic-gam...
2,Fred KERLEY,USA,9.81,SB,0.108,NaN,https://worldathletics.org/results/olympic-gam...
3,Akani SIMBINE,RSA,9.82,NR,0.149,NaN,https://worldathletics.org/results/olympic-gam...
4,Lamont Marcell JACOBS,ITA,9.85,SB,0.114,NaN,https://worldathletics.org/results/olympic-gam...
5,Letsile TEBOGO,BOT,9.86,NR,0.178,NaN,https://worldathletics.org/results/olympic-gam...
6,Kenneth BEDNAREK,USA,9.88,NaN,0.163,NaN,https://worldathletics.org/results/olympic-gam...
7,Oblique SEVILLE,JAM,9.91,NaN,0.171,NaN,https://worldathletics.org/results/olympic-gam...
0,Noah LYLES,USA,9.83,=WL,0.145,0.0,https://worldathletics.org/results/world-athle...
1,Letsile TEBOGO,BOT,9.88,NR,0.129,0.0,https://worldathletics.org/results/world-athle...
